<a href="https://colab.research.google.com/github/roaddaniel-blip/Actividad-5-/blob/main/Actividad_5_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ***Análisis de sentimiento***

Este código contiene un proyecto aplicado en el que selecciona un dataset público, ya sea de clasificación o regresión, realiza su preparación y versionamiento, entrena un modelo de aprendizaje automático, ajusta hiperparámetros y registra los experimentos utilizando MLflow.

### 1) Carga de dataset

Cargamos el dataset con el que estaremos trabajando. Se carga desde un estorno local.


In [ ]:
import pandas as pd

# Cargar el archivo de Excel
file_path = '/content/Actividad 5 .xlsx'
df = pd.read_excel(file_path)

# Mostrar las primeras filas del archivo
display(df.head())

,text,label
0,"No funciona como se describe, muy grosero con ...",NEG
1,El envío fue extremadamente maravilloso y el a...,POS
2,El envío fue extremadamente fantástico y el ar...,POS
3,"Me encanta, funciona de maravilla y el diseño ...",POS
4,El envío fue extremadamente maravilloso y el a...,POS


### 2) Limpieza de datos
Vamos a realizar los siguientes pasos de limpieza:
1. Verificar si existen valores nulos.
2. Eliminar registros duplicados.
3. Limpiar y normalizar el texto (convertir a minúsculas y quitar espacios en blanco innecesarios).

In [ ]:
# 1. Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())

# 2. Eliminar filas con valores nulos (si las hubiera)
df = df.dropna().reset_index(drop=True)

# 3. Eliminar duplicados
duplicados_antes = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nRegistros duplicados eliminados: {duplicados_antes}")

Valores nulos por columna:
text     0
label    0
dtype: int64

Registros duplicados eliminados: 443


In [ ]:
# 4. Limpieza básica de la columna 'text' (minúsculas y remoción de espacios innecesarios)
df['text_clean'] = df['text'].astype(str).str.lower().str.strip()

# Mostrar el DataFrame limpio
display(df.head())

,text,label,text_clean
0,"No funciona como se describe, muy grosero con ...",NEG,"no funciona como se describe, muy grosero con ..."
1,El envío fue extremadamente maravilloso y el a...,POS,el envío fue extremadamente maravilloso y el a...
2,El envío fue extremadamente fantástico y el ar...,POS,el envío fue extremadamente fantástico y el ar...
3,"Me encanta, funciona de maravilla y el diseño ...",POS,"me encanta, funciona de maravilla y el diseño ..."
4,"No funciona como se describe, muy malo con la ...",NEG,"no funciona como se describe, muy malo con la ..."


### 3) Preparación de datos
Estandarizaremos las etiquetas de texto a valores numéricos (0 y 1) y representaremos el texto usando TF-IDF para poder alimentar los modelos de Machine Learning.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# 1. Codificar etiquetas (POS -> 1, NEG -> 0)
le = LabelEncoder()
y = le.fit_transform(df['label'])

# 2. Vectorizar el texto limpio usando TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['text_clean'])

# 3. Dividir en conjuntos de entrenamiento y prueba (80% entreno, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")

Dimensiones de X_train: (45, 85)
Dimensiones de X_test: (12, 85)


### 4) Entrenamiento y Evaluación de Modelos
Entrenaremos y compararemos dos algoritmos de clasificación:
1. **Regresión Logística**
2. **Random Forest**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- Modelo 1: Regresión Logística ---
print("=== REGRESIÓN LOGÍSTICA ===")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

# --- Modelo 2: Random Forest ---
print("\n=== RANDOM FOREST ===")
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

=== REGRESIÓN LOGÍSTICA ===
Exactitud (Accuracy): 1.0000
              precision    recall  f1-score   support

         NEG       1.00      1.00      1.00         5
         POS       1.00      1.00      1.00         7

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12


=== RANDOM FOREST ===
Exactitud (Accuracy): 1.0000
              precision    recall  f1-score   support

         NEG       1.00      1.00      1.00         5
         POS       1.00      1.00      1.00         7

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



### 5) Optimización con Validación Cruzada e hiperparámetros (Grid Search)
Utilizaremos `GridSearchCV` con validación cruzada de 5 pliegues para ajustar los parámetros de ambos modelos optimizando la métrica estandarizada F1-macro.

In [ ]:
from sklearn.model_selection import GridSearchCV

# --- 1. Ajuste de Regresión Logística ---
print("=== Optimización de Regresión Logística ===")
lr_params = {
    'C': [0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['liblinear', 'lbfgs']
}
grid_lr = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid=lr_params,
    scoring='f1_macro',
    cv=5
)
grid_lr.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_lr.best_params_}")
print(f"Mejor F1-macro en CV: {grid_lr.best_score_:.4f}")

# --- 2. Ajuste de Random Forest ---
print("\n=== Optimización de Random Forest ===")
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    scoring='f1_macro',
    cv=5
)
grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Mejor F1-macro en CV: {grid_rf.best_score_:.4f}")

=== Optimización de Regresión Logística ===
Mejores parámetros: {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'}
Mejor F1-macro en CV: 0.9766

=== Optimización de Random Forest ===
Mejores parámetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Mejor F1-macro en CV: 0.9544


### 6) Evaluación final

In [ ]:
# Evaluar los mejores modelos en el conjunto de prueba independiente
best_lr = grid_lr.best_estimator_
best_rf = grid_rf.best_estimator_

print("=== Desempeño Final Regresión Logística (Optimizado) ===")
y_pred_opt_lr = best_lr.predict(X_test)
print(classification_report(y_test, y_pred_opt_lr, target_names=le.classes_))

print("\n=== Desempeño Final Random Forest (Optimizado) ===")
y_pred_opt_rf = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_opt_rf, target_names=le.classes_))

=== Desempeño Final Regresión Logística (Optimizado) ===
              precision    recall  f1-score   support

         NEG       1.00      1.00      1.00         5
         POS       1.00      1.00      1.00         7

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12


=== Desempeño Final Random Forest (Optimizado) ===
              precision    recall  f1-score   support

         NEG       1.00      1.00      1.00         5
         POS       1.00      1.00      1.00         7

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



### 7) Registro en MLflow
Instalaremos la biblioteca `mlflow` y registraremos los parámetros óptimos encontrados con GridSearchCV, junto con las métricas finales obtenidas en el conjunto de prueba independiente para ambos modelos.

In [ ]:
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 438.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import mlflow
from sklearn.metrics import precision_recall_fscore_support

# Configurar experimento local de MLflow
mlflow.set_experiment("Clasificacion_Sentimiento_Actividad_5")

# --- 1. Registrar Regresión Logística ---
with mlflow.start_run(run_name="Logistic_Regression_Optimized"):
    # Registrar parámetros
    mlflow.log_params(grid_lr.best_params_)

    # Calcular métricas globales de test
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_opt_lr, average='macro')
    acc = accuracy_score(y_test, y_pred_opt_lr)

    # Registrar métricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1)
    mlflow.log_metric("precision_macro", precision)
    mlflow.log_metric("recall_macro", recall)

    # Guardar modelo
    mlflow.sklearn.log_model(best_lr, artifact_path="model")
    print("Resultados de Regresión Logística registrados en MLflow.")

# --- 2. Registrar Random Forest ---
with mlflow.start_run(run_name="Random_Forest_Optimized"):
    # Registrar parámetros
    mlflow.log_params(grid_rf.best_params_)

    # Calcular métricas globales de test
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_opt_rf, average='macro')
    acc = accuracy_score(y_test, y_pred_opt_rf)

    # Registrar métricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1)
    mlflow.log_metric("precision_macro", precision)
    mlflow.log_metric("recall_macro", recall)

    # Guardar modelo autorizando los tipos de árbol requeridos para evadir la alerta de skops
    mlflow.sklearn.log_model(
        best_rf,
        artifact_path="model",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )
    print("Resultados de Random Forest registrados en MLflow.")

2026/10/19 02:07:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/19 02:08:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Resultados de Regresión Logística registrados en MLflow.
Resultados de Random Forest registrados en MLflow.


### 8) Consulta de Experimentos en MLflow
.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
import os

# Permitir explícitamente el uso de almacenamiento en archivos locales (evita el error de MLflowException)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Definir las posibles rutas de tracking donde se pudieron guardar los experimentos
tracking_uris = ["./mlruns", "sqlite:///mlflow.db"]

for uri in tracking_uris:
    print(f"🔍 Buscando experimentos en el backend de tracking: '{uri}'...")
    mlflow.set_tracking_uri(uri)
    client = MlflowClient()

    try:
        experiments = client.search_experiments()
    except Exception as e:
        print(f"   ⚠️ No se pudo acceder a {uri}: {e}\n")
        continue

    valid_experiments = [e for e in experiments if e.name != "Default" or len(client.search_runs(experiment_ids=[e.experiment_id])) > 0]

    if valid_experiments:
        print(f"   ✅ Se encontraron {len(valid_experiments)} experimento(s) con actividad:\n")
        for exp in valid_experiments:
            print(f"   =========================================================================")
            print(f"   🧪 EXPERIMENTO: {exp.name} (ID: {exp.experiment_id})")
            print(f"   =========================================================================")

            runs = client.search_runs(experiment_ids=[exp.experiment_id])
            for run in runs:
                run_id = run.info.run_id
                run_name = run.data.tags.get("mlflow.runName", "N/A")
                status = run.info.status

                print(f"      📌 Run Name: {run_name} (ID: {run_id})")
                print(f"         Estado: {status}")

                # Mostrar Parámetros
                params = run.data.params
                print("         ⚙️ Parámetros:")
                if params:
                    for k, v in params.items():
                        print(f"            - {k}: {v}")
                else:
                    print("            - No hay parámetros registrados.")

                # Mostrar Métricas
                metrics = run.data.metrics
                print("         📊 Métricas:")
                if metrics:
                    for k, v in metrics.items():
                        print(f"            - {k}: {round(v, 4)}")
                else:
                    print("            - No hay métricas registradas.")

                # Mostrar Artefactos
                try:
                    artifacts = client.list_artifacts(run_id)
                    print("         📦 Artefactos:")
                    if artifacts:
                        for artifact in artifacts:
                            print(f"            - RUTA: {artifact.path} (Es Directorio: {artifact.is_dir})")
                    else:
                        print("            - No hay artefactos registrados.")
                except Exception as artifact_error:
                    print(f"         📦 Error al leer artefactos: {artifact_error}")
                print("      " + "-" * 50 + "\n")
    else:
        print("   ❌ No se encontraron experimentos activos con ejecuciones en este backend.\n")

🔍 Buscando experimentos en el backend de tracking: './mlruns'...
   ❌ No se encontraron experimentos activos con ejecuciones en este backend.

🔍 Buscando experimentos en el backend de tracking: 'sqlite:///mlflow.db'...
   ❌ No se encontraron experimentos activos con ejecuciones en este backend.



In [ ]:
import mlflow
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load data
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

# Enable autologging
mlflow.sklearn.autolog()

with mlflow.start_run():
    model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Model scoring is automatically captured
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)

2026/10/19 02:25:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
import mlflow
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split

# Load data
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42
)

# Enable autologging
mlflow.sklearn.autolog(max_tuning_runs=10)

# Define parameter grid
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
}

with mlflow.start_run(run_name="RF Hyperparameter Tuning"):
    rf = RandomForestClassifier(random_state=42)
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_score = grid_search.score(X_test, y_test)
    print(f"Best params: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.3f}")
    print(f"Test score: {best_score:.3f}")

2026/10/19 02:28:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/19 02:28:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/19 02:28:20 INFO mlflow.sklearn.utils: Logging the 10 best runs, 26 runs will be omitted.


Best params: {'max_depth': 15, 'min_samples_split': 5, 'n_estimators': 200}
Best CV score: 0.975
Test score: 0.981


In [ ]:
!pip install -q optuna

import mlflow
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

# Load data
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

mlflow.sklearn.autolog()


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
    }

    with mlflow.start_run(nested=True):
        model = GradientBoostingClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        accuracy = model.score(X_test, y_test)
        return accuracy


with mlflow.start_run():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_accuracy", study.best_value)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 1.6 MB/s eta 0:00:00


[I 2026-10-19 02:31:41,807] A new study created in memory with name: no-name-bc7c2779-40a8-440c-8c3f-c8844dbde004
2026/10/19 02:31:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-10-19 02:31:51,090] Trial 0 finished with value: 0.956140350877193 and parameters: {'n_estimators': 159, 'max_depth': 5, 'learning_rate': 0.03896823543577068}. Best is trial 0 with value: 0.956140350877193.
2026/10/19 02:31:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

In [ ]:
# 1. Instalar librerías necesarias (usando localtunnel en lugar de pyngrok para evitar tokens)
!pip install mlflow --quiet
!npm install -g localtunnel --quiet

import mlflow
import subprocess
import time

# 2. Crear un experimento de prueba
mlflow.set_experiment("MLflow Quickstart")

# 3. Registrar una ejecución (Run) con métricas ficticias
with mlflow.start_run(run_name="mi-primer-entrenamiento"):
    # Parámetros del modelo
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("epochs", 10)
    mlflow.log_param("model_type", "LogisticRegression")

    # Métricas de rendimiento
    mlflow.log_metric("accuracy", 0.89)
    mlflow.log_metric("loss", 0.24)

print("¡Datos de prueba registrados con éxito!")

# 4. Iniciar el servidor de MLflow en segundo plano en el puerto 5000
print("Iniciando el servidor de MLflow...")
subprocess.Popen(["mlflow", "ui", "--port", "5000"])
time.sleep(3) # Esperar un momento a que el servidor se levante

# 5. Obtener la IP pública de este entorno Colab (necesaria como contraseña de seguridad para localtunnel)
print("\n--- CONTRASEÑA DE SEGURIDAD PARA LOCALTUNNEL ---")
print("Copia la siguiente dirección IP, la necesitarás para ingresar al enlace de abajo:")
!curl ipv4.icanhazip.com
print("--------------------------------------------------\n")

# 6. Crear el túnel público mediante localtunnel
!lt --port 5000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 4s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦¡Datos de prueba registrados con éxito!
Iniciando el servidor de MLflow...

--- CONTRASEÑA DE SEGURIDAD PARA LOCALTUNNEL ---
Copia la siguiente dirección IP, la necesitarás para ingresar al enlace de abajo:
136.108.78.176
--------------------------------------------------

your url is: https://dark-ideas-hammer.loca.lt
^C
